# 📊 AdventureWorks — Analysis

**تحليل تفاعلي لمبيعات AdventureWorks 2019**

---

## 📋 نظرة عامة

نوت بوك مخصص للتحليل السريع — يقرأ من ملف CSV جاهز (نتيجة ETL سابق).

**الميزة**: لا اتصال بقاعدة البيانات — التحميل في **3 ثوانٍ**.

---

## 🗂️ المحتويات

| # | القسم |
|---|---|
| 1 | تحميل البيانات |
| 2 | فحص سريع |
| 3 | المقاييس (KPIs) |
| 4 | Filter Context |
| 5 | تحليل التخفيضات |

---

## 🔗 ملفات المشروع

- `ETL.ipynb` — خط أنابيب ETL الكامل
- `dashboard.py` — لوحة Streamlit
- `output/adventureworks.csv` — البيانات

---

**المؤلف**: بسام طارق

In [1]:
# ============================================================
# الخلية 1: تحميل البيانات من CSV
# ============================================================

import pandas as pd

df = pd.read_csv("output/adventureworks.csv")
df["OrderDate"] = pd.to_datetime(df["OrderDate"])

print(f"✅ تم التحميل: {len(df):,} صف، {len(df.columns)} عمود")
print(f"📅 الفترة: {df['OrderDate'].min().date()} → {df['OrderDate'].max().date()}")

✅ تم التحميل: 121,317 صف، 21 عمود
📅 الفترة: 2011-05-30 → 2014-06-29


In [2]:
# ============================================================
# الخلية 2: فحص سريع
# ============================================================

print("📊 ملخص البيانات\n")
print(f"   الصفوف:          {len(df):,}")
print(f"   الأعمدة:         {len(df.columns)}")
print(f"   العملاء:         {df['CustomerID'].nunique():,}")
print(f"   المنتجات:        {df['ProductID'].nunique():,}")
print(f"   الفئات:          {df['CategoryName'].nunique()}")
print(f"   السنوات:         {sorted(df['OrderYear'].unique().tolist())}")

# --- NULLs ---
nulls = df.isnull().sum().sum()
print(f"\n🔍 إجمالي NULLs: {nulls}")

# --- الرابحة vs الترويجية ---
print(f"\n📈 توزيع الربحية:")
print(f"   ✅ رابحة:    {df['IsProfitable'].sum():,} ({df['IsProfitable'].mean()*100:.1f}%)")
print(f"   ⚠️ ترويجية: {(~df['IsProfitable']).sum():,} ({(~df['IsProfitable']).mean()*100:.1f}%)")

📊 ملخص البيانات

   الصفوف:          121,317
   الأعمدة:         21
   العملاء:         19,119
   المنتجات:        266
   الفئات:          4
   السنوات:         [2011, 2012, 2013, 2014]

🔍 إجمالي NULLs: 60398

📈 توزيع الربحية:
   ✅ رابحة:    92,156 (76.0%)
   ⚠️ ترويجية: 29,161 (24.0%)


In [3]:
# ============================================================
# الخلية 3: المقاييس (مكافئ DAX Measures)
# ============================================================

def safe_divide(a, b):
    """قسمة آمنة — مكافئ DIVIDE() في DAX"""
    return a / b if b else 0


# --- المقاييس الأساسية ---
def total_sales(d):       return d["LineTotal"].sum()
def total_profit(d):      return d["Profit"].sum()
def total_qty(d):         return d["OrderQty"].sum()
def order_count(d):       return d["SalesOrderID"].nunique()
def unique_customers(d):  return d["CustomerID"].nunique()

# --- المقاييس النسبية ---
def profit_margin(d):
    return safe_divide(total_profit(d), total_sales(d)) * 100

def avg_order_value(d):
    return safe_divide(total_sales(d), order_count(d))

def avg_qty_per_order(d):
    return safe_divide(total_qty(d), order_count(d))


# --- عرض KPIs ---
def show_kpis(data, title="KPIs"):
    print(f"\n{'=' * 50}")
    print(f"📈 {title}")
    print("=" * 50)
    print(f"   💰 إجمالي المبيعات:     {total_sales(data):>15,.0f}")
    print(f"   💵 إجمالي الربح:        {total_profit(data):>15,.0f}")
    print(f"   📊 هامش الربح:          {profit_margin(data):>14.2f}%")
    print(f"   🧾 عدد الطلبات:         {order_count(data):>15,}")
    print(f"   👥 عدد العملاء:         {unique_customers(data):>15,}")
    print(f"   📦 إجمالي الكمية:       {total_qty(data):>15,}")
    print(f"   🛒 متوسط قيمة الطلب:    {avg_order_value(data):>15,.2f}")
    print(f"   📦 متوسط الكمية/طلب:    {avg_qty_per_order(data):>15,.2f}")
    print("=" * 50)


# --- التشغيل ---
show_kpis(df, "كل البيانات")


📈 كل البيانات
   💰 إجمالي المبيعات:         109,846,381
   💵 إجمالي الربح:              9,371,904
   📊 هامش الربح:                    8.53%
   🧾 عدد الطلبات:                  31,465
   👥 عدد العملاء:                  19,119
   📦 إجمالي الكمية:               274,914
   🛒 متوسط قيمة الطلب:           3,491.07
   📦 متوسط الكمية/طلب:               8.74


In [4]:
# ============================================================
# الخلية 4: Filter Context
# ============================================================

# --- حسب الفئة ---
show_kpis(df[df["CategoryName"] == "Bikes"], "الدراجات فقط")
show_kpis(df[df["CategoryName"] == "Clothing"], "الملابس فقط")

# --- حسب السنة ---
show_kpis(df[df["OrderYear"] == 2013], "سنة 2013 فقط")
show_kpis(df[df["OrderYear"] == 2014], "سنة 2014 فقط")

# --- حسب القناة ---
show_kpis(df[df["OnlineOrderFlag"] == 0], "طلبات مندوبين")
show_kpis(df[df["OnlineOrderFlag"] == 1], "طلبات أونلاين")


📈 الدراجات فقط
   💰 إجمالي المبيعات:          94,651,173
   💵 إجمالي الربح:              7,936,394
   📊 هامش الربح:                    8.38%
   🧾 عدد الطلبات:                  18,368
   👥 عدد العملاء:                   9,719
   📦 إجمالي الكمية:                90,268
   🛒 متوسط قيمة الطلب:           5,153.05
   📦 متوسط الكمية/طلب:               4.91

📈 الملابس فقط
   💰 إجمالي المبيعات:           2,120,543
   💵 إجمالي الربح:                308,882
   📊 هامش الربح:                   14.57%
   🧾 عدد الطلبات:                   9,877
   👥 عدد العملاء:                   7,327
   📦 إجمالي الكمية:                73,670
   🛒 متوسط قيمة الطلب:             214.70
   📦 متوسط الكمية/طلب:               7.46

📈 سنة 2013 فقط
   💰 إجمالي المبيعات:          43,652,266
   💵 إجمالي الربح:              3,367,898
   📊 هامش الربح:                    7.72%
   🧾 عدد الطلبات:                  14,234
   👥 عدد العملاء:                  11,141
   📦 إجمالي الكمية:               131,926
   🛒 متوسط قيمة الطلب:       

In [5]:
# ============================================================
# الخلية 5: تحليل التخفيضات الموسمية
# ============================================================

# --- حسب السنة ---
yearly = df.groupby("OrderYear").agg(
    إجمالي=("IsProfitable", "count"),
    رابحة=("IsProfitable", "sum"),
).reset_index()

yearly["ترويجية"] = yearly["إجمالي"] - yearly["رابحة"]
yearly["نسبة_ترويجية_%"] = (yearly["ترويجية"] / yearly["إجمالي"] * 100).round(1)

print("📅 التخفيضات حسب السنة:\n")
print(yearly.to_string(index=False))

# --- حسب الشهر ---
monthly = df.groupby("OrderMonth").agg(
    إجمالي=("IsProfitable", "count"),
    ترويجية=("IsProfitable", lambda x: (~x).sum()),
).reset_index()

monthly["نسبة_%"] = (monthly["ترويجية"] / monthly["إجمالي"] * 100).round(1)
monthly = monthly.sort_values("نسبة_%", ascending=False)

print("\n📅 أعلى 5 أشهر بالتخفيضات:\n")
print(monthly.head(5).to_string(index=False))

📅 التخفيضات حسب السنة:

 OrderYear  إجمالي  رابحة  ترويجية  نسبة_ترويجية_%
      2011    6588   2913     3675            55.8
      2012   20827  11065     9762            46.9
      2013   56711  45162    11549            20.4
      2014   37191  33016     4175            11.2

📅 أعلى 5 أشهر بالتخفيضات:

 OrderMonth  إجمالي  ترويجية  نسبة_%
          9   10710     3123    29.2
          6   11436     3317    29.0
          3   12098     3220    26.6
         12    9726     2498    25.7
         10   10413     2616    25.1


In [11]:
# ============================================================
# الخلية 6: تحليل المنتجات الخاسرة
# ============================================================

print("🔍 المنتجات الأكثر خسارة (بالقيمة الإجمالية):\n")

# --- تجميع حسب المنتج ---
by_product = df.groupby(["ProductID", "ProductName"]).agg(
    إجمالي_المبيعات=("LineTotal", "sum"),
    إجمالي_الربح=("Profit", "sum"),
    عدد_الصفوف=("Profit", "count"),
    صفوف_خاسرة=("IsProfitable", lambda x: (~x).sum()),
).reset_index()

by_product["هامش_%"] = (by_product["إجمالي_الربح"] / by_product["إجمالي_المبيعات"] * 100).round(2)

# --- أسوأ 10 منتجات ---
worst_10 = by_product.nsmallest(10, "إجمالي_الربح")
print(worst_10[["ProductName", "إجمالي_المبيعات", "إجمالي_الربح", "هامش_%"]].to_string(index=False))

print("\n" + "=" * 60)
print("🔍 المنتجات الأكثر ربحية:\n")
best_10 = by_product.nlargest(10, "إجمالي_الربح")
print(best_10[["ProductName", "إجمالي_المبيعات", "إجمالي_الربح", "هامش_%"]].to_string(index=False))

🔍 المنتجات الأكثر خسارة (بالقيمة الإجمالية):

               ProductName  إجمالي_المبيعات   إجمالي_الربح  هامش_%
          Road-650 Red, 44     9.433406e+05 -153696.116612  -16.29
   Touring-1000 Yellow, 60     1.518133e+06 -132745.719453   -8.74
          Road-650 Red, 60     1.016529e+06  -69800.112786   -6.87
   Touring-1000 Yellow, 46     1.426373e+06  -62974.720176   -4.42
        Road-650 Black, 52     1.042910e+06  -61914.203970   -5.94
Long-Sleeve Logo Jersey, L     1.987550e+05  -54986.266240  -27.67
          Road-650 Red, 62     8.798279e+05  -42967.775357   -4.88
        Road-650 Black, 58     8.696328e+05  -40995.266149   -4.71
          Road-650 Red, 48     8.786667e+05  -40235.397140   -4.58
     Touring-3000 Blue, 50     3.485815e+05  -39493.573285  -11.33

🔍 المنتجات الأكثر ربحية:

            ProductName  إجمالي_المبيعات  إجمالي_الربح  هامش_%
 Mountain-200 Black, 42     4.009495e+06 674216.578641   16.82
 Mountain-200 Black, 38     4.400593e+06 673444.470300   15.30
 

In [12]:
# ============================================================
# الخلية 7: تحليل الفئات — رابحة أم خاسرة؟
# ============================================================

print("📊 الأداء حسب الفئة:\n")

by_cat = df.groupby("CategoryName").agg(
    إجمالي_المبيعات=("LineTotal", "sum"),
    إجمالي_الربح=("Profit", "sum"),
    صفوف_خاسرة=("IsProfitable", lambda x: (~x).sum()),
    إجمالي_الصفوف=("IsProfitable", "count"),
).reset_index()

by_cat["هامش_%"] = (by_cat["إجمالي_الربح"] / by_cat["إجمالي_المبيعات"] * 100).round(2)
by_cat["نسبة_خاسرة_%"] = (by_cat["صفوف_خاسرة"] / by_cat["إجمالي_الصفوف"] * 100).round(1)

by_cat = by_cat.sort_values("إجمالي_الربح", ascending=False)
print(by_cat.to_string(index=False))

📊 الأداء حسب الفئة:

CategoryName  إجمالي_المبيعات  إجمالي_الربح  صفوف_خاسرة  إجمالي_الصفوف  هامش_%  نسبة_خاسرة_%
       Bikes     9.465117e+07  7.936394e+06       19340          40031    8.38          48.3
 Accessories     1.272073e+06  6.363943e+05           0          41194   50.03           0.0
  Components     1.180259e+07  4.902332e+05        4769          18698    4.15          25.5
    Clothing     2.120543e+06  3.088822e+05        5052          21394   14.57          23.6


In [13]:
# ============================================================
# الخلية 8: التركيز — أي منتج مسؤول عن أكبر خسارة؟
# ============================================================

# --- إجمالي الخسائر ---
total_loss = df[df["Profit"] < 0]["Profit"].sum()
print(f"💰 إجمالي الخسائر: {total_loss:,.0f}\n")

# --- أكبر 5 منتجات مسؤولة عن الخسائر ---
losses_by_product = (df[df["Profit"] < 0]
                     .groupby("ProductName")["Profit"]
                     .sum()
                     .nsmallest(5))

print("🔻 أكبر 5 منتجات مسؤولة عن الخسائر:\n")
for product, loss in losses_by_product.items():
    pct = (loss / total_loss) * 100
    print(f"   • {product:<40} {loss:>15,.0f}  ({pct:.1f}%)")

# --- هل تتركز الخسائر في منتج واحد؟ ---
top_loss = losses_by_product.iloc[0]
top_pct = (top_loss / total_loss) * 100
print(f"\n📌 أعلى منتج مسؤول عن {top_pct:.1f}% من إجمالي الخسائر")

💰 إجمالي الخسائر: -5,034,388

🔻 أكبر 5 منتجات مسؤولة عن الخسائر:

   • Touring-1000 Yellow, 60                         -259,044  (5.1%)
   • Road-250 Black, 44                              -243,354  (4.8%)
   • Touring-1000 Yellow, 46                         -218,141  (4.3%)
   • Road-250 Black, 48                              -209,488  (4.2%)
   • Road-650 Red, 44                                -173,953  (3.5%)

📌 أعلى منتج مسؤول عن 5.1% من إجمالي الخسائر


In [14]:
# ============================================================
# الخلية 9: تحليل المناطق (Territory)
# ============================================================

print("🗺️ الأداء حسب المنطقة:\n")

by_territory = df.groupby("TerritoryID").agg(
    إجمالي_المبيعات=("LineTotal", "sum"),
    إجمالي_الربح=("Profit", "sum"),
    عدد_العملاء=("CustomerID", "nunique"),
).reset_index()

by_territory["هامش_%"] = (by_territory["إجمالي_الربح"] / by_territory["إجمالي_المبيعات"] * 100).round(2)

# --- أعلى 5 وأدنى 5 ---
print("🏆 أعلى 5 مناطق:")
print(by_territory.nlargest(5, "إجمالي_الربح").to_string(index=False))

print("\n⚠️ أدنى 5 مناطق:")
print(by_territory.nsmallest(5, "إجمالي_الربح").to_string(index=False))

🗺️ الأداء حسب المنطقة:

🏆 أعلى 5 مناطق:
 TerritoryID  إجمالي_المبيعات  إجمالي_الربح  عدد_العملاء  هامش_%
           9     1.065534e+07  3.434255e+06         3625   32.23
           4     2.418461e+07  1.563711e+06         4565    6.47
           1     1.608494e+07  1.364602e+06         3428    8.48
          10     7.670721e+06  1.231909e+06         1951   16.06
           8     4.915408e+06  1.043017e+06         1812   21.22

⚠️ أدنى 5 مناطق:
 TerritoryID  إجمالي_المبيعات   إجمالي_الربح  عدد_العملاء  هامش_%
           2     6.939374e+06 -269493.184095           57   -3.88
           5     7.879655e+06 -151057.419049           91   -1.92
           3     7.909009e+06 -142450.752728           69   -1.80
           6     1.635577e+07  404316.404762         1677    2.47
           7     7.251556e+06  893094.679726         1844   12.32


In [15]:
# ============================================================
# الخلية 10: لماذا المناطق 2، 5، 3 خاسرة؟
# ============================================================

loss_territories = [2, 5, 3]

print("🔍 تحليل المناطق الخاسرة (2, 5, 3):\n")

for tid in loss_territories:
    territory_data = df[df["TerritoryID"] == tid]
    
    avg_discount = territory_data["UnitPriceDiscount"].mean() * 100
    avg_qty = territory_data["OrderQty"].mean()
    margin = (territory_data["Profit"].sum() / territory_data["LineTotal"].sum()) * 100
    orders = territory_data["SalesOrderID"].nunique()
    
    print(f"📍 المنطقة {tid}:")
    print(f"   عدد الطلبات:      {orders:,}")
    print(f"   متوسط الخصم:      {avg_discount:.1f}%")
    print(f"   متوسط الكمية:     {avg_qty:.1f}")
    print(f"   الهامش:           {margin:.2f}%")
    print()

# --- مقارنة مع المنطقة الرابحة ---
print("=" * 55)
print("📊 مقارنة: المنطقة 9 (رابحة) vs 2 (خاسرة)\n")

for tid, label in [(9, "رابحة"), (2, "خاسرة")]:
    t = df[df["TerritoryID"] == tid]
    print(f"📍 المنطقة {tid} ({label}):")
    print(f"   متوسط الخصم:   {t['UnitPriceDiscount'].mean()*100:.1f}%")
    print(f"   متوسط الكمية:  {t['OrderQty'].mean():.1f}")
    print(f"   متوسط السعر:   ${t['UnitPrice'].mean():,.0f}")
    print()

🔍 تحليل المناطق الخاسرة (2, 5, 3):

📍 المنطقة 2:
   عدد الطلبات:      352
   متوسط الخصم:      0.4%
   متوسط الكمية:     3.4
   الهامش:           -3.88%

📍 المنطقة 5:
   عدد الطلبات:      486
   متوسط الخصم:      0.5%
   متوسط الكمية:     3.2
   الهامش:           -1.92%

📍 المنطقة 3:
   عدد الطلبات:      385
   متوسط الخصم:      0.5%
   متوسط الكمية:     3.3
   الهامش:           -1.80%

📊 مقارنة: المنطقة 9 (رابحة) vs 2 (خاسرة)

📍 المنطقة 9 (رابحة):
   متوسط الخصم:   0.1%
   متوسط الكمية:  1.2
   متوسط السعر:   $653

📍 المنطقة 2 (خاسرة):
   متوسط الخصم:   0.4%
   متوسط الكمية:  3.4
   متوسط السعر:   $425



In [16]:
# ============================================================
# الخلية 11: مزيج المنتجات لكل منطقة
# ============================================================

print("🔍 مزيج الفئات في كل منطقة:\n")

for tid in [9, 2, 5, 3]:
    t = df[df["TerritoryID"] == tid]
    
    # --- مزيج الفئات ---
    mix = t.groupby("CategoryName").agg(
        صفوف=("IsProfitable", "count"),
        مبيعات=("LineTotal", "sum"),
        ربح=("Profit", "sum"),
    )
    mix["نسبة_%"] = (mix["صفوف"] / mix["صفوف"].sum() * 100).round(1)
    
    print(f"\n📍 المنطقة {tid} ({'رابحة' if tid == 9 else 'خاسرة'}):")
    print(f"{'الفئة':<15} {'النسبة %':>10} {'المبيعات':>15} {'الربح':>15}")
    print("-" * 60)
    for cat, row in mix.iterrows():
        print(f"{cat:<15} {row['نسبة_%']:>10} {row['مبيعات']:>15,.0f} {row['ربح']:>15,.0f}")

🔍 مزيج الفئات في كل منطقة:


📍 المنطقة 9 (رابحة):
الفئة             النسبة %        المبيعات           الربح
------------------------------------------------------------
Accessories           47.7         162,638          95,796
Bikes                 34.8      10,175,871       3,297,243
Clothing              14.3         113,176          27,516
Components             3.2         203,651          13,701

📍 المنطقة 2 (خاسرة):
الفئة             النسبة %        المبيعات           الربح
------------------------------------------------------------
Accessories            8.3          51,002          17,686
Bikes                 41.9       5,675,571        -331,170
Clothing              18.8         161,204          13,873
Components            30.9       1,051,598          30,118

📍 المنطقة 5 (خاسرة):
الفئة             النسبة %        المبيعات           الربح
------------------------------------------------------------
Accessories            8.3          45,749          15,933
Bikes          

In [18]:
# ============================================================
# الخلية 12: هل المنتج خاسر في كل المناطق أم في منطقة معينة؟
# ============================================================

# --- اختر منتجًا يظهر في القائمة الخاسرة ---
target_product = "Road-250 Black, 44"

print(f"🔍 تحليل: {target_product}\n")

prod_data = df[df["ProductName"] == target_product]

by_territory = prod_data.groupby("TerritoryID").agg(
    عدد_الصفوف=("Profit", "count"),
    متوسط_السعر=("UnitPrice", "mean"),
    التكلفة=("StandardCost", "mean"),
    الربح=("Profit", "sum"),
    هامش=("Profit", lambda x: (x.sum() / prod_data.loc[x.index, "LineTotal"].sum() * 100)),
).round(2)

print(by_territory.to_string())

# --- السعر والتكلفة ---
print(f"\n💰 السعر المعروض (ListPrice): ${prod_data['ListPrice'].iloc[0]:,.0f}")
print(f"💰 التكلفة المعيارية:         ${prod_data['StandardCost'].iloc[0]:,.0f}")
print(f"💰 الفرق (الهامش الممكن):      ${prod_data['ListPrice'].iloc[0] - prod_data['StandardCost'].iloc[0]:,.0f}")

🔍 تحليل: Road-250 Black, 44

             عدد_الصفوف  متوسط_السعر  التكلفة     الربح   هامش
TerritoryID                                                   
1                    50      1706.16  1554.95  -6344.61  -2.98
2                    50      1373.16  1554.95 -38316.50 -14.55
3                    54      1390.38  1554.95 -24492.83 -11.73
4                   141      1564.85  1554.95 -44151.21  -7.87
5                    38      1390.46  1554.95 -19897.69 -13.03
6                    98      1475.89  1554.95 -41446.88 -11.18
7                    58      1910.45  1554.95  10235.55   5.20
8                    39      2197.22  1554.95  24870.93  28.06
9                   105      2331.16  1554.95  81501.78  33.30
10                   72      1970.68  1554.95  21674.33   9.96

💰 السعر المعروض (ListPrice): $2,443
💰 التكلفة المعيارية:         $1,555
💰 الفرق (الهامش الممكن):      $888


In [19]:
# ============================================================
# الخلية 13: القرار — أي المناطق تحتاج تدخل تسعير؟
# ============================================================

print("🎯 توصيات التسعير حسب المنطقة:\n")

pricing_check = df.groupby("TerritoryID").agg(
    متوسط_سعر_البيع=("UnitPrice", "mean"),
    متوسط_التكلفة=("StandardCost", "mean"),
    إجمالي_المبيعات=("LineTotal", "sum"),
    إجمالي_الربح=("Profit", "sum"),
).reset_index()

pricing_check["نسبة_السعر_للتكلفة_%"] = (
    pricing_check["متوسط_سعر_البيع"] / pricing_check["متوسط_التكلفة"] * 100
).round(1)

pricing_check["الحالة"] = pricing_check["إجمالي_الربح"].apply(
    lambda x: "🔴 خاسرة" if x < 0 else "🟢 رابحة"
)

pricing_check = pricing_check.sort_values("نسبة_السعر_للتكلفة_%")

print(pricing_check.to_string(index=False))

print("\n" + "=" * 60)
print("📋 التوصية:")
print("=" * 60)

worst = pricing_check.head(3)["TerritoryID"].tolist()
print(f"🔴 المناطق {worst} تبيع بأقل من 90% من التكلفة — تحتاج تدخل فوري")

🎯 توصيات التسعير حسب المنطقة:

 TerritoryID  متوسط_سعر_البيع  متوسط_التكلفة  إجمالي_المبيعات  إجمالي_الربح  نسبة_السعر_للتكلفة_%  الحالة
           2       425.339749     439.364817     6.939374e+06 -2.694932e+05                  96.8 🔴 خاسرة
           3       460.012164     469.485210     7.909009e+06 -1.424508e+05                  98.0 🔴 خاسرة
           5       486.293912     494.075476     7.879655e+06 -1.510574e+05                  98.4 🔴 خاسرة
           6       359.150804     322.106516     1.635577e+07  4.043164e+05                 111.5 🟢 رابحة
           4       459.079027     375.812424     2.418461e+07  1.563711e+06                 122.2 🟢 رابحة
           1       449.407377     362.901647     1.608494e+07  1.364602e+06                 123.8 🟢 رابحة
           7       435.731356     323.127126     7.251556e+06  8.930947e+05                 134.8 🟢 رابحة
          10       457.406884     331.035447     7.670721e+06  1.231909e+06                 138.2 🟢 رابحة
           8   